# LoRA Fine-tuning: Llama-3.2-1B -- Banka Musteri Hizmetleri

Project 5'te sifirdan yazdigimiz LoRA kodunu **gercek bir LLM'e** uyguluyoruz.

**Senaryo:** Banka musterilerinin sorularini cevaplayan bir chatbot egitiyoruz.  
Dataset: `bitext/Bitext-customer-support-llm-chatbot-training-dataset` -- 26.000 musteri-hizmetleri ornek

**Pipeline:**
1. Llama-3.2-1B yukle
2. Baseline inference -- banka sorusuna ilk cevap
3. `inject_lora()` -- sadece ~%1 parametre egitilecek
4. 500 musteri-hizmetleri ornegi ile fine-tune
5. Fine-tuned inference -- fark ne kadar?
6. `merge_lora_weights()` -- adapteri baz modele birlestir

> **Onemi:** Runtime > Change runtime type > **T4 GPU** sec, sonra calistir

In [ ]:
# Kurulum -- her calistirmada temiz clone
!pip install -q transformers datasets accelerate huggingface_hub
!rm -rf /content/ai-lora-finetuning
!git clone https://github.com/Mertgdk/ai-lora-finetuning.git
import sys
sys.path.insert(0, '/content/ai-lora-finetuning')
print('Kurulum tamamlandi.')

In [ ]:
import torch
import math
import types
import importlib
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import login

# importlib.reload -- cache sorununu onler
import src.core.lora as _lora_mod
importlib.reload(_lora_mod)
from src.core.lora import inject_lora, count_parameters, LoRALayer

import src.core.merge as _merge_mod
importlib.reload(_merge_mod)
from src.core.merge import merge_lora_weights

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Hugging Face login
# Token: https://huggingface.co/settings/tokens > New token > Read erisimi
# Format: hf_xxxxxxxxxxxxxxxxxxxx
HF_TOKEN = 'hf_BURAYA_KENDI_TOKENINI_YAZ'
login(token=HF_TOKEN)
print('HF login basarili.')

In [ ]:
# Model yukle -- device_map='auto' yok (inject_lora ile uyumsuz)
MODEL_ID = 'meta-llama/Llama-3.2-1B'
print(f'{MODEL_ID} yukleniyor... (~2.5GB, birkac dakika surebilir)')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)
model = model.to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'Model yuklendi. Toplam parametre: {total_params:,}')
print(f'Device: {next(model.parameters()).device}')

In [ ]:
# Baseline inference -- fine-tuning ONCESI cevaplari kaydet
TEMPLATE = (
    'You are a helpful bank customer support agent. '
    'Answer the customer question clearly and professionally.\n\n'
    '### Customer:\n{instruction}\n\n### Support Agent:\n'
)

TEST_QUESTIONS = [
    'My account has been blocked. How can I unblock it?',
    'I want to apply for a credit card. What are the requirements?',
    'I was charged twice for the same transaction. What should I do?',
]

def generate(mdl, tok, prompt, max_new_tokens=100):
    inputs = tok(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    generated = out[0][inputs['input_ids'].shape[1]:]
    return tok.decode(generated, skip_special_tokens=True).strip()

print('=== BASELINE (Fine-tuning oncesi) ===')
baseline_answers = {}
for q in TEST_QUESTIONS:
    ans = generate(model, tokenizer, TEMPLATE.format(instruction=q))
    baseline_answers[q] = ans
    print(f'Musteri: {q}')
    print(f'Cevap:   {ans[:200]}')
    print('-' * 60)

In [ ]:
# LoRA inject + T4 bfloat16 patch
inject_lora(model, target_modules=['q_proj', 'v_proj'], rank=8, alpha=16)

# T4 GPU native bfloat16 matmul desteklemiyor
# Fix: x'i float32'ye upcast et, hesapla, geri bfloat16 cast et
def _lora_fwd(self, x):
    return ((x.float() @ self.A @ self.B) * self.scaling).to(x.dtype)

patched = 0
for _, m in model.named_modules():
    if isinstance(m, LoRALayer):
        m.forward = types.MethodType(_lora_fwd, m)
        patched += 1

model = model.to(device)
params_after = count_parameters(model)

print(f'Toplam:      {params_after["total"]:>15,}')
print(f'Egitilecek:  {params_after["trainable"]:>15,}  ({params_after["trainable_pct"]:.4f}%)')
print(f'Dondurulmus: {params_after["frozen"]:>15,}')
print(f'{patched} LoRA katmani float32 patch uygulandi.')

cpu_p = [n for n, p in model.named_parameters() if p.device.type == 'cpu']
print('Tum parametreler GPU\'da.' if not cpu_p else f'UYARI: {len(cpu_p)} parametre CPU\'da!')

In [ ]:
# Dataset
class CustomerSupportDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=256):
        self.tokenizer = tokenizer
        self.max_length = max_length
        tmpl = (
            'You are a helpful bank customer support agent. '
            'Answer the customer question clearly and professionally.\n\n'
            '### Customer:\n{instruction}\n\n### Support Agent:\n{response}'
        )
        self.samples = [
            tmpl.format(instruction=item['instruction'], response=item['response'])
            for item in data
            if item.get('instruction', '').strip() and item.get('response', '').strip()
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.samples[idx],
            truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt',
        )
        return {'input_ids': enc['input_ids'].squeeze(), 'attention_mask': enc['attention_mask'].squeeze()}

print('Dataset indiriliyor...')
raw = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset', split='train[:500]')
dataset = CustomerSupportDataset(raw, tokenizer)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
print(f'Dataset: {len(dataset)} ornek, {len(dataloader)} batch')
print(f'Ornek -- {raw[0]["instruction"]}')

In [ ]:
# Egitim
EPOCHS = 3
LR = 2e-4

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
all_losses, epoch_losses = [], []

model.train()
print(f'Egitim: {EPOCHS} epoch | lr={LR} | batch=4 | {params_after["trainable"]:,} param egitilecek')
print('-' * 60)

for epoch in range(EPOCHS):
    epoch_loss, n = 0.0, 0
    for i, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = input_ids.clone()
        labels[labels == tokenizer.pad_token_id] = -100

        loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        all_losses.append(loss.item())
        n += 1
        if (i + 1) % 20 == 0:
            print(f'  Epoch {epoch+1} | Batch {i+1}/{len(dataloader)} | Loss: {loss.item():.4f}')

    avg = epoch_loss / n
    epoch_losses.append(avg)
    print(f'Epoch {epoch+1}/{EPOCHS} -- Ort. Loss: {avg:.4f}')
    print('-' * 60)

print(f'Loss: {epoch_losses[0]:.4f} -> {epoch_losses[-1]:.4f} ({100*(epoch_losses[0]-epoch_losses[-1])/epoch_losses[0]:.1f}% dusus)')

In [ ]:
# Loss grafigi
plt.figure(figsize=(10, 4))
plt.plot(all_losses, alpha=0.4, color='steelblue', label='Batch loss')
bpe = len(dataloader)
for i, el in enumerate(epoch_losses):
    plt.axhline(el, xmin=i*bpe/len(all_losses), xmax=(i+1)*bpe/len(all_losses),
                color=['red','orange','green'][i], linewidth=2.5, label=f'Epoch {i+1}: {el:.4f}')
plt.xlabel('Batch'); plt.ylabel('Loss')
plt.title('LoRA Fine-tuning -- Banka Musteri Hizmetleri (Llama-3.2-1B)')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ONCESI vs SONRASI
model.eval()
print('=' * 60)
print('FINE-TUNING ONCESI vs SONRASI')
print('=' * 60)
for q in TEST_QUESTIONS:
    new_ans = generate(model, tokenizer, TEMPLATE.format(instruction=q))
    print(f'\nMusteri: {q}')
    print(f'ONCESI:  {baseline_answers[q][:200]}')
    print(f'SONRASI: {new_ans[:200]}')
    print('-' * 60)

In [ ]:
# Merge
test_input = tokenizer(TEMPLATE.format(instruction=TEST_QUESTIONS[0]), return_tensors='pt').to(device)
model.eval()
with torch.no_grad():
    logits_before = model(**test_input).logits.clone()

merge_lora_weights(model)

with torch.no_grad():
    logits_after = model(**test_input).logits

max_diff = (logits_before - logits_after).abs().max().item()
params_merged = count_parameters(model)

print(f'Max logit farki:  {max_diff:.2e}')
print(f'Merge:            {"BASARILI" if max_diff < 1e-2 else "BASARISIZ"}')
print(f'Trainable:        {params_merged["trainable_pct"]:.2f}% (LoRA gitti)')
print()
print(f'Model:       Llama-3.2-1B ({total_params:,} parametre)')
print(f'Egitilen:    {params_after["trainable"]:,} ({params_after["trainable_pct"]:.4f}%)')
print(f'Loss:        {epoch_losses[0]:.4f} -> {epoch_losses[-1]:.4f}')
print(f'Merge diff:  {max_diff:.2e}')